In [25]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

In [29]:
import pandas as pd
import numpy as np

bank_statement = pd.read_excel("C:/Users/dkirungu.ICEALIONGROUP/Downloads/May Bank Statement.xlsx", 
                               header=7)
cashbook_sheets = pd.read_excel("C:/Users/dkirungu.ICEALIONGROUP/Downloads/May Cashbook 2026.xlsx", 
                         header=10, sheet_name=None)
print(cashbook_sheets.items())
def get_cashbook_data(cashbook_sheets):
    cashbook = pd.DataFrame()
    for sheet_name, df in cashbook_sheets.items():
        df['Sheet Name'] = sheet_name
        cashbook = pd.concat([cashbook, df], ignore_index=True)
    return cashbook
cashbook = get_cashbook_data(cashbook_sheets)

dict_items([('Sheet1',            Document No_ Posting Date                    Description  \
0             F243610-P   2026-05-01  JALLY AND BARA MOTORS LIMITED   
1             F243684-P   2026-05-01             ACE AUTOCENTRE LTD   
2             F243685-P   2026-05-01             ACE AUTOCENTRE LTD   
3             F243495-P   2026-05-01     INTEGRATED MOTOR ASSESSORS   
4             F243615-P   2026-05-01           SOLEX MOTORS LIMITED   
...                 ...          ...                            ...   
1943        GJ000000103   2026-05-31                            NaN   
1944  PV-2026-000000299   2026-05-31                            NaN   
1945  PV-2026-000000300   2026-05-31                            NaN   
1946  PV-2026-000000301   2026-05-31                            NaN   
1947  PV-2026-000000302   2026-05-31                       NCBA LTD   

     External Document No_  Payment Id     Amount  Unnamed: 6  Class  \
0                      NaN         NaN  -46897.50   

In [30]:
cashbook['Amount'].sum() # - 47274585.62

np.float64(69889079.81999993)

In [31]:
def clean_bankstatement(bank_statement):
        current_bank_df = bank_statement.copy()
        
        df2 = current_bank_df[['Transaction Details','Posting Date', 'Value Date','Bank Reference', 'Debit Amount','Credit Amount']]
       # 1. Extract the cheque number where CHQ exists (returns NaN where it doesn't match)
        extracted_chq = df2["Transaction Details"].str.extract(r"CHQ-?(\d+)\b")[0]

        # 2. Use np.select to conditionally fill the new column
        conditions = [
        df2["Transaction Details"].str.contains("CHQ", na=False)
        & extracted_chq.notna()
        ]
        choices = [extracted_chq]

        # If condition is met, use choices; otherwise, default to 'Bank Reference'
        df2["Clean_Cheque_No"] = np.select(conditions, choices, default=df2["Bank Reference"])

        df2['Clean_Cheque_No'] = df2['Clean_Cheque_No'].fillna('No Cheque No')

        return df2

clean_bank_df = clean_bankstatement(bank_statement)

In [32]:
grouped_bank = clean_bank_df.groupby(['Clean_Cheque_No','Transaction Details','Posting Date','Value Date','Bank Reference']).agg(
 Debit_Amount = ('Debit Amount','sum'),
 Credit_Amount = ('Credit Amount','sum')
).reset_index()

grouped_cashbook = cashbook.groupby(['External Document No_', 'Balance Account', 'Source No_',
       'Balance account Name']).agg(
           Amount = ('Amount','sum'),
           Description = ('Description','first'),
           Posting_Date = ('Posting Date','first'),
           Claim_No_ = ('Claim No_','first'),
           Document_No_ = ('Document No_', lambda x: ', '.join(x.dropna().astype(str))),
            Policy_No_ = ('Policy No_', lambda x: ', '.join(x.dropna().astype(str)))
).reset_index()


merged_df = pd.merge(grouped_cashbook, grouped_bank, 
                     left_on='External Document No_', 
                     right_on='Clean_Cheque_No',
                      how='outer', 
                      indicator=True,
                      suffixes=('_cashbook', '_bank')) 
merged_df['Bank_Amount'] = merged_df['Credit_Amount'].fillna(0) - merged_df['Debit_Amount'].fillna(0)
merged_df['Amount_Difference'] = merged_df['Amount'].fillna(0) - merged_df['Bank_Amount'].fillna(0)

In [34]:
merged_df.to_excel("C:/Users/dkirungu.ICEALIONGROUP/Downloads/merged_output2.xlsx", index=False)

matching_items = merged_df[merged_df['_merge'] == 'both']
unclearing_items = merged_df[merged_df['_merge'] == 'left_only']
unreceipted_items = merged_df[merged_df['_merge'] == 'right_only']
unreceipted_items
to_manually_resolve = matching_items[matching_items['Amount_Difference'] != 0]['Clean_Cheque_No'].tolist()
manual_interv_df = matching_items[matching_items['Clean_Cheque_No'].isin(to_manually_resolve)]
matching_items = matching_items[~matching_items['Clean_Cheque_No'].isin(to_manually_resolve)]


# with pd.ExcelWriter("C:/Users/dkirungu.ICEALIONGROUP/Downloads/BANKRECON.xlsx") as writer:
#     matching_items.to_excel(writer, sheet_name='Matching Items', index=False)
#     unclearing_items.to_excel(writer, sheet_name='Unclearing Items', index=False)
#     unreceipted_items.to_excel(writer, sheet_name='Unreceipted Items', index=False)
#     manual_interv_df.to_excel(writer, sheet_name='Manual Intervention', index=False)

In [35]:
#saving original unclearing
cashbook[cashbook['External Document No_'].isin(unreceipted_items['External Document No_'])].to_excel("C:/Users/dkirungu.ICEALIONGROUP/Downloads/original_unclearing.xlsx", index=False)

In [8]:
##bouncedcheques

grouped_bank = clean_bank_df.groupby('Clean_Cheque_No').agg(
    Total_Debit = ('Debit Amount', 'sum'),  
    Total_Credit = ('Credit Amount', 'sum')
).reset_index()

bounced_cheques = grouped_bank[(grouped_bank['Total_Debit'] > 0) & (grouped_bank['Total_Credit'] > 0)]

bounced_cheques
bounced_cheques_data = merged_df[merged_df['Clean_Cheque_No'].isin(bounced_cheques['Clean_Cheque_No'])]
bounced_cheques_data.to_excel("C:/Users/dkirungu.ICEALIONGROUP/Downloads/bounced_cheques.xlsx", index=False)
# bounced_cheques_data.to_excel("C:/Users/dkirungu.ICEALIONGROUP/Downloads/bounced_cheques.xlsx", index=False)

In [24]:
cleaned_unclearing = unreceipted_items.drop_duplicates(subset=['Clean_Cheque_No'], keep='first')
cleaned_unclearing['Amount'].sum()

np.float64(0.0)

In [19]:
cleaned_unclearing

In [38]:
print(cashbook['Amount'].sum()) 
print(bank_statement['Credit Amount'].sum() - bank_statement['Debit Amount'].sum())

69889079.81999993
20750437.96999991


In [45]:
69889079.81999993 +  47274585.62

# 47274585.62 - 117163665.44

117163665.43999994